In [7]:
# =========================
# IMPORTS
# =========================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import joblib

# =========================
# LOAD DATASET
# =========================
dataset = pd.read_csv("dataset/stroke.csv")
print("✅ Dataset Loaded:", dataset.shape)

# =========================
# DEFINE COLUMNS
# =========================
numeric_cols = ['age', 'avg_glucose_level', 'bmi']

categorical_cols = [
    'gender', 'hypertension', 'heart_disease',
    'ever_married', 'work_type',
    'Residence_type', 'smoking_status'
]

# =========================
# HANDLE MISSING VALUES
# =========================
for col in numeric_cols:
    dataset[col] = dataset[col].fillna(dataset[col].mean())

# =========================
# ONE-HOT ENCODING
# =========================
dataset = pd.get_dummies(dataset, columns=categorical_cols)

# =========================
# SPLIT DATA
# =========================
y = dataset['stroke']
X = dataset.drop('stroke', axis=1)

# ✅ SAVE FEATURE NAMES
feature_columns = X.columns.tolist()

# =========================
# SCALE NUMERIC
# =========================
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# =========================
# TRAIN TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

# =========================
# BALANCE DATA
# =========================
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# =========================
# TRAIN MODEL
# =========================
model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_bal, y_train_bal)

# =========================
# SAVE FILES
# =========================
joblib.dump(model, "stroke_model.pkl")
joblib.dump(scaler, "stroke_scaler.pkl")
joblib.dump(feature_columns, "stroke_features.pkl")

print("🔥 Stroke model saved successfully!")

# =========================================================
# 🔥 FINAL PREDICT FUNCTION (MATCHES YOUR INPUT FORMAT)
# =========================================================
def predict_stroke(new_data: dict):

    model = joblib.load("stroke_model.pkl")
    scaler = joblib.load("stroke_scaler.pkl")
    cols = joblib.load("stroke_features.pkl")

    numeric_cols = ['age', 'avg_glucose_level', 'bmi']

    # Convert input → DataFrame
    df = pd.DataFrame([new_data])

    # One-hot encode (same as training)
    df = pd.get_dummies(df)

    # Add missing columns
    for col in cols:
        if col not in df:
            df[col] = 0

    # Keep correct order
    df = df[cols]

    # Scale numeric columns
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    # Predict
    prob = model.predict_proba(df)[0][1] * 100

    result = {
        "probability": round(prob, 2),
        "chance_level": (
            "High" if prob >= 70 else
            "Medium" if prob >= 30 else
            "Low"
        )
    }

    print("🎯 Result:", result)
    return result


# =========================
# ✅ TEST (YOUR FORMAT)
# =========================
predict_stroke({
    "gender": "Male",
    "age": 67,
    "hypertension": 1,
    "heart_disease": 0,
    "ever_married": "Yes",
    "work_type": "Private",
    "Residence_type": "Urban",
    "avg_glucose_level": 228,
    "bmi": 36,
    "smoking_status": "formerly smoked"
})

✅ Dataset Loaded: (5110, 12)
🔥 Stroke model saved successfully!
🎯 Result: {'probability': np.float64(22.0), 'chance_level': 'Low'}


{'probability': np.float64(22.0), 'chance_level': 'Low'}

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
import joblib

In [9]:
dataset = pd.read_csv("dataset/stroke.csv")
print("✅ Dataset Loaded:", dataset.shape)

dataset.head()

✅ Dataset Loaded: (5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [10]:
numeric_cols = ['age', 'avg_glucose_level', 'bmi']

categorical_cols = [
    'gender', 'hypertension', 'heart_disease',
    'ever_married', 'work_type',
    'Residence_type', 'smoking_status'
]

In [11]:
for col in numeric_cols:
    dataset[col] = dataset[col].fillna(dataset[col].mean())

In [12]:
dataset = pd.get_dummies(dataset, columns=categorical_cols)

dataset.head()

,id,age,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,gender_Other,hypertension_0,hypertension_1,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,9046,67.0,228.69,36.600000,1,False,True,False,True,False,...,False,True,False,False,False,True,False,True,False,False
1,51676,61.0,202.21,28.893237,1,True,False,False,True,False,...,False,False,True,False,True,False,False,False,True,False
2,31112,80.0,105.92,32.500000,1,False,True,False,True,False,...,False,True,False,False,True,False,False,False,True,False
3,60182,49.0,171.23,34.400000,1,True,False,False,True,False,...,False,True,False,False,False,True,False,False,False,True
4,1665,79.0,174.12,24.000000,1,True,False,False,False,True,...,False,False,True,False,True,False,False,False,True,False


In [13]:
y = dataset['stroke']
X = dataset.drop('stroke', axis=1)

# Save feature names
feature_columns = X.columns.tolist()

print("✅ Total Features:", len(feature_columns))

✅ Total Features: 24


In [14]:
scaler = StandardScaler()

X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

In [16]:
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", pd.Series(y_train_bal).value_counts())

Before SMOTE: stroke
0    3270
1     153
Name: count, dtype: int64
After SMOTE: stroke
0    3270
1    3270
Name: count, dtype: int64


In [17]:
model = RandomForestClassifier(
    n_estimators=150,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_bal, y_train_bal)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [18]:
joblib.dump(model, "stroke_model.pkl")
joblib.dump(scaler, "stroke_scaler.pkl")
joblib.dump(feature_columns, "stroke_features.pkl")

print("🔥 All files saved!")

🔥 All files saved!


In [19]:
def predict_stroke(new_data: dict):

    model = joblib.load("stroke_model.pkl")
    scaler = joblib.load("stroke_scaler.pkl")
    cols = joblib.load("stroke_features.pkl")

    numeric_cols = ['age', 'avg_glucose_level', 'bmi']

    # Convert input → DataFrame
    df = pd.DataFrame([new_data])

    # One-hot encode
    df = pd.get_dummies(df)

    # Add missing columns
    for col in cols:
        if col not in df:
            df[col] = 0

    # Remove extra columns
    df = df[cols]

    # Scale numeric only
    df[numeric_cols] = scaler.transform(df[numeric_cols])

    # Predict
    prob = model.predict_proba(df)[0][1] * 100

    result = {
        "probability": round(prob, 2),
        "chance_level": (
            "High" if prob >= 70 else
            "Medium" if prob >= 30 else
            "Low"
        )
    }

    return result

In [20]:
predict_stroke({
    "gender": "Male",
    "age": 67,
    "hypertension": 1,
    "heart_disease": 0,
    "ever_married": "Yes",
    "work_type": "Private",
    "Residence_type": "Urban",
    "avg_glucose_level": 228,
    "bmi": 36,
    "smoking_status": "formerly smoked"
})

{'probability': np.float64(22.0), 'chance_level': 'Low'}